In [1]:
# Install pdemtools, localtileserver, and folium
!pip install -q pdemtools localtileserver folium

import pdemtools as pdt
import os
import folium
from localtileserver import TileClient, get_folium_tile_layer

# Define the bounding box in EPSG:3413 (meters)
bounds = (3173848, -903621, 3262186, -810502)  # xmin, ymin, xmax, ymax for Saint Petersburg

# Download the ArcticDEM mosaic at 2m resolution
dem = pdt.load.mosaic(
    dataset='arcticdem',
    resolution=2,  # Highest resolution: 2 meters
    bounds=bounds,
    version='v4.1'  # Latest version
)

# Create output directory if it doesn't exist
if not os.path.exists('output'):
    os.mkdir('output')

# Save the DEM as a GeoTIFF
output_path = 'output/saint_petersburg_arcticdem_2m.tif'
dem.rio.to_raster(
    output_path,
    compress='ZSTD',  # Compression for smaller file size
    predictor=3,
    zlevel=1
)

print(f'DEM saved to {output_path}')

# Visualize with Leaflet via folium and localtileserver
client = TileClient(output_path)

# Create a folium tile layer with a colormap suitable for elevation
tile_layer = get_folium_tile_layer(
    client,
    colormap='terrain',  # Good for elevation data
    vmin=0,              # Adjust based on expected min elevation (e.g., sea level)
    vmax=200             # Adjust based on expected max elevation in Saint Petersburg
)

# Create the map centered on the raster, with OpenStreetMap basemap
m = folium.Map(
    location=client.center(),
    zoom_start=10,
    tiles='openstreetmap'
)

# Add the DEM layer
m.add_child(tile_layer)

# Add layer control
folium.LayerControl().add_to(m)

# Display the map
m

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.1 MB/s eta 0:00:00
ERROR: Exc

ModuleNotFoundError: No module named 'pdemtools'

In [ ]:
# ==============================================================================
# 1. УСТАНОВКА И ИМПОРТ
# ==============================================================================
!pip install -q pystac-client planetary-computer rioxarray geopandas folium matplotlib rasterio

import os
import numpy as np
import geopandas as gpd
import rioxarray
import pystac_client
import planetary_computer
import folium
from shapely.geometry import box
from matplotlib import cm
from matplotlib.colors import Normalize

# Настройки
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CRS_ARCTIC = "EPSG:3413"  # Исходная проекция ArcticDEM
CRS_MAP = "EPSG:3857"     # Проекция для веб-карт (Google/OSM)

# ==============================================================================
# 2. ОПРЕДЕЛЕНИЕ ОБЛАСТИ (Ваши координаты EPSG:3413)
# ==============================================================================
# xmin, ymin, xmax, ymax (из вашего скрипта)
bounds_3413 = (3173848, -903621, 3262186, -810502) 

print("📍 Подготовка геометрии...")
# Создаем полигон в исходной проекции
bbox_poly = gpd.GeoDataFrame(
    {'geometry': [box(*bounds_3413)]}, 
    crs=CRS_ARCTIC
)

# Для поиска в каталоге STAC нужны координаты Lat/Lon (WGS84)
bbox_wgs84 = bbox_poly.to_crs("EPSG:4326").total_bounds
print(f"   Поиск в координатах WGS84: {bbox_wgs84}")

# ==============================================================================
# 3. ПОИСК ДАННЫХ (STAC API)
# ==============================================================================
print("\n📡 Поиск 2-метровых стрипов ArcticDEM...")
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Ищем в коллекции 'arctic-dem-strips' (это сырые данные 2м разрешения)
search = catalog.search(
    collections=["arctic-dem-strips"],
    bbox=bbox_wgs84,
    sortby=[{'field': 'properties.eo:cloud_cover', 'direction': 'asc'}], # Сортируем по облачности
    max_items=5
)

items = list(search.items())
print(f"   Найдено снимков: {len(items)}")

if not items:
    raise ValueError("Снимки на эту область не найдены через API.")

# Берем лучший снимок (первый в списке)
best_item = items[0]
print(f"   📥 Загружаем лучший снимок: {best_item.id} (Облачность: {best_item.properties['eo:cloud_cover']}%)")

# ==============================================================================
# 4. ЗАГРУЗКА И ОБРАБОТКА
# ==============================================================================
href = best_item.assets["data"].href

# Открываем лениво и обрезаем по нашей рамке
ds = rioxarray.open_rasterio(href, masked=True)
ds_clipped = ds.rio.clip_box(*bounds_3413, crs=CRS_ARCTIC)

# Сохраняем оригинал (GeoTIFF)
dem_path = os.path.join(OUTPUT_DIR, 'saint_petersburg_arcticdem_2m.tif')
ds_clipped.rio.to_raster(dem_path, compress='ZSTD')
print(f"✅ DEM сохранен: {dem_path}")

# ==============================================================================
# 5. ВИЗУАЛИЗАЦИЯ (FOLIUM)
# ==============================================================================
print("\n🗺️ Генерация карты...")

# Для Folium нужно перепроецировать в EPSG:3857 (Web Mercator)
# Иначе карта "уедет" или будет искажена
ds_web = ds_clipped.rio.reproject(CRS_MAP)

# Удаляем выбросы (нодата) для красивой картинки
data = ds_web.values[0]
data = np.where(data < -100, np.nan, data) # Фильтр ошибок

# Создаем цветовую карту (Colormap) для наложения
# Folium ImageOverlay не понимает сырые float данные, ему нужна картинка (RGB)
norm = Normalize(vmin=np.nanmin(data), vmax=np.nanmax(data))
cmap = cm.get_cmap('terrain')
rgba_img = cmap(norm(data)) # Конвертируем высоты в цвета

# Делаем прозрачным фон (где нет данных)
rgba_img[:, :, 3] = np.where(np.isnan(data), 0, 0.7) # Alpha канал 0.7

# Получаем границы картинки для Folium
bounds_img = [
    [ds_web.rio.bounds()[1], ds_web.rio.bounds()[0]], # miny, minx (наоборот для Folium)
    [ds_web.rio.bounds()[3], ds_web.rio.bounds()[2]]  # maxy, maxx
]

# Центр карты
center = [(bounds_img[0][0] + bounds_img[1][0])/2, (bounds_img[0][1] + bounds_img[1][1])/2]

m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')

# Накладываем изображение
folium.raster_layers.ImageOverlay(
    image=rgba_img,
    bounds=bounds_img,
    opacity=0.8,
    name='ArcticDEM 2m'
).add_to(m)

folium.LayerControl().add_to(m)

# Сохраняем HTML
m.save(os.path.join(OUTPUT_DIR, 'map_2m.html'))
print("✅ Карта готова. Отображаем...")
m